# IT Operations Intelligence
## Fase 1-4: Comprensión del Negocio y Exploración Inicial de Datos

**Objetivo:** Analizar el dataset de incidencias IT para comprender su estructura, identificar patrones de fallos y preparar los datos para fases posteriores de modelado predictivo.


### Fase 1. Comprensión del Negocio

**Problema:** Los equipos de soporte IT gestionan miles de tickets con diferentes prioridades, categorías y niveles de cumplimiento de SLA.

**Objetivo:** Desarrollar un modelo predictivo capaz de identificar tickets con alto riesgo de incumplimiento de SLA, permitiendo una gestión proactiva de las incidencias.

**Pregunta de negocio:** ¿Es posible predecir qué tickets de soporte no cumplirán con el SLA antes de que sea demasiado tarde?

---

### Fase 2. Comprensión de los Datos


In [ ]:
import pandas as pd # Manipulación y análisis de datos en DataFrames
import numpy as np # Operaciones numéricas y álgebra lineal
import os # Gestión de rutas y directorios del sistema
import matplotlib.pyplot as plt # Creación de gráficos estáticos
import seaborn as sns # Visualización estadística avanzada

# Muestra todas las columnas del DataFrame sin truncar
pd.set_option("display.max_columns", None)

# Limita la visualización a 100 filas como máximo
pd.set_option("display.max_rows", 100)

# Ruta donde se almacenarán los gráficos generados durante el análisis
ruta_graficos = "../reports/graphics"

# Crea el directorio de gráficos si aún no existe (exist_ok=True evita error si ya existe)
os.makedirs(ruta_graficos, exist_ok=True)


In [ ]:
# Carga del dataset de incidencias IT desde un archivo CSV en la ruta especificada
df = pd.read_csv("../data/raw/incident_event_log.csv")


In [ ]:
# Muestra las primeras 5 filas del DataFrame para una primera inspección visual de los datos
df.head()


##### Análisis de las primeras filas del dataset

Las primeras 5 filas nos permiten observar la estructura general del registro de incidencias. Se pueden identificar columnas como el identificador del ticket (`number`), fechas de apertura y cierre (`opened_at`, `resolved_at`, `closed_at`), métricas de severidad (`priority`, `impact`, `urgency`), la variable objetivo (`made_sla`), y campos descriptivos como `assignment_group`, `category` y `contact_type`. La diversidad de tipos de datos (numéricos, categóricos, fechas) anticipa la necesidad de un procesamiento diferenciado en la fase de preparación.


In [ ]:
# Muestra las últimas 5 filas del DataFrame para verificar cómo finalizan los registros
df.tail()


##### Análisis de las últimas filas del dataset

Las últimas filas del dataset permiten comprobar si existe algún orden cronológico en los registros. Si el dataset está ordenado por fecha de apertura, las últimas filas corresponderían a las incidencias más recientes. También se verifica que no haya filas con valores atípicos o faltantes al final del archivo que pudieran indicar problemas en la carga de datos.


In [ ]:
# Toma una muestra aleatoria de 5 filas para una inspección imparcial del contenido
df.sample(5)


##### Análisis de muestra aleatoria del dataset

La muestra aleatoria complementa la inspección de cabecera y cola, reduciendo el sesgo de posición. Confirmamos que los valores en todas las columnas son consistentes con los observados anteriormente y que no hay anomalías evidentes en registros intermedios. La muestra aleatoria es útil para detectar patrones que no son visibles solo con head() y tail().


In [ ]:
# Imprime el número total de filas del DataFrame
print(f"Filas: {df.shape[0]}")

# Imprime el número total de columnas del DataFrame
print(f"Columnas: {df.shape[1]}")


##### Dimensiones del dataset

El dataset contiene **141712 filas** (incidencias registradas) y **36 columnas** (variables descriptivas). Este volumen de datos es adecuado para un análisis exploratorio robusto y para la construcción de modelos predictivos, ya que proporciona suficiente variabilidad estadística sin llegar a ser computacionalmente intensivo.


In [ ]:
# Muestra un resumen completo del DataFrame: número de entradas, tipo de cada columna y uso de memoria
df.info()


##### Tipos de datos y uso de memoria

El resumen de `info()` revela la composición del dataset en términos de tipos de datos. Las columnas numéricas (int64, float64) son adecuadas para análisis estadísticos directos, mientras que las columnas de tipo object requieren codificación para modelos predictivos. Las columnas de fecha deben convertirse al tipo datetime para extraer características temporales. El uso de memoria es moderado, lo que permite trabajar con el dataset completo en memoria RAM sin necesidad de técnicas de muestreo o procesamiento distribuido.


In [ ]:
# Obtiene la lista completa de nombres de columnas del DataFrame
df.columns.tolist()


### Fase 3. Preparación / Limpieza de Datos


In [ ]:
# .T transpone el resultado para facilitar la lectura (cada columna es una fila)
df.describe().T


##### Análisis de estadísticas descriptivas

El resumen estadístico muestra la distribución de las variables numéricas. Se observan los valores medios, la dispersión (desviación estándar) y el rango (mínimo-máximo) de cada variable. Es importante verificar que los valores mínimos y máximos sean coherentes con el dominio del problema. Por ejemplo, variables como `priority` deberían estar en un rango esperado (1-5 típicamente). La presencia de valores atípicos puede detectarse comparando la media con la mediana (percentil 50%).


In [ ]:
# Calcula la suma de valores nulos por cada columna
nulos = df.isnull().sum()

# Imprime el encabezado del reporte de nulos
print("Valores nulos por columna:")

# Verifica si existe al menos un valor nulo en todo el DataFrame
if nulos.sum() > 0:

    # Muestra solo las columnas que contienen valores nulos
    print(nulos[nulos > 0])

else:
    # Informa que no se encontraron valores nulos
    print("No se detectaron valores nulos.")


##### Análisis de valores nulos

La verificación de valores nulos es un paso crítico en la limpieza de datos. La ausencia de valores nulos simplifica significativamente el preprocesamiento, ya que no será necesario aplicar técnicas de imputación o eliminación de registros. Sin embargo, es importante confirmar que los valores faltantes no hayan sido codificados como cadenas vacías o valores especiales (como 'NA' o '-1') que pasarían desapercibidos para `isnull()`.


In [ ]:
# Cuenta el número de filas duplicadas en el DataFrame
dup = df.duplicated().sum()

# Imprime la cantidad de registros duplicados encontrados
print(f"Registros duplicados: {dup}")


##### Análisis de registros duplicados

La ausencia de registros duplicados indica que el dataset no contiene entradas redundantes que podrían sesgar los análisis estadísticos o los modelos predictivos. Los duplicados suelen ser el resultado de errores en la extracción de datos o en la concatenación de múltiples fuentes, por lo que su inexistencia es una señal positiva sobre la calidad de los datos.


In [ ]:
# Muestra el tipo de datos (dtype) de cada columna del DataFrame
df.dtypes


In [ ]:
# Selecciona únicamente las columnas numéricas (int64, float64)
numericas = df.select_dtypes(include=np.number)

# Selecciona únicamente las columnas categóricas (tipo object o string)
categoricas = df.select_dtypes(include="object")

# Imprime la cantidad de variables numéricas detectadas
print(f"Variables numéricas: {len(numericas.columns)}")

# Imprime la cantidad de variables categóricas detectadas
print(f"Variables categóricas: {len(categoricas.columns)}")


##### Separación de variables numéricas y categóricas

La separación entre variables numéricas y categóricas es fundamental para aplicar las técnicas de análisis y preprocesamiento adecuadas a cada tipo. Las variables numéricas se prestan a análisis de correlación, escalado y detección de outliers. Las variables categóricas requieren codificación (one-hot, label encoding) para ser utilizadas en modelos de machine learning.


In [ ]:
# Muestra la lista de nombres de las columnas numéricas identificadas
numericas.columns.tolist()


In [ ]:
# Muestra la lista de nombres de las columnas categóricas identificadas
categoricas.columns.tolist()


### Fase 4. Análisis Exploratorio Inicial


In [ ]:
# Imprime el encabezado del análisis de la variable objetivo
print("Distribución de made_sla (cumplimiento de SLA):")

# Cuenta cuántos tickets cumplieron (True) y no cumplieron (False) el SLA
print(df["made_sla"].value_counts())

# Imprime una línea en blanco para separar la salida
print()

# Imprime el encabezado de la sección de porcentajes
print("Porcentaje:")

# Calcula y muestra el porcentaje de cada categoría redondeado a 2 decimales
print(round(df["made_sla"].value_counts(normalize=True) * 100, 2))


##### Análisis de la variable objetivo: `made_sla`

La variable `made_sla` indica si un ticket cumplió (True) o no (False) con el Acuerdo de Nivel de Servicio. Un desbalanceo significativo entre clases es común en problemas de SLA, donde la mayoría de tickets suelen cumplir con el acuerdo. Este desbalanceo deberá ser abordado en la fase de modelado mediante técnicas como sobremuestreo, submuestreo o uso de métricas adecuadas (F1-score, precisión-recall en lugar de exactitud).


In [ ]:
# Define el tamaño de la figura para la matriz de correlación
plt.figure(figsize=(10, 8))

# Calcula la matriz de correlación de Pearson entre todas las variables numéricas
corr = numericas.corr()

# Dibuja el heatmap con mapa de colores coolwarm, mostrando valores y formato de 2 decimales
sns.heatmap(corr, cmap="coolwarm", annot=True, fmt=".2f", linewidths=0.5)

# Establece el título del gráfico
plt.title("Matriz de Correlación - Variables Numéricas")

# Ajusta automáticamente los subplots para que encajen en el área de la figura
plt.tight_layout()

# Guarda el gráfico en la ruta definida con alta resolución (300 dpi)
plt.savefig(f"{ruta_graficos}/Nb01_01_matriz_correlacion.png", dpi=300, bbox_inches="tight")

# Muestra el gráfico en pantalla
plt.show()


##### Análisis de la matriz de correlación

La matriz de correlación revela las relaciones lineales entre las variables numéricas. Los valores cercanos a +1 indican una fuerte correlación positiva (ambas variables aumentan juntas), mientras que valores cercanos a -1 indican correlación negativa (una aumenta cuando la otra disminuye). Las correlaciones altas entre variables predictoras pueden indicar multicolinealidad, lo que podría afectar a modelos lineales como regresión logística. Es especialmente relevante observar qué variables se correlacionan fuertemente con `made_sla`, la variable objetivo.


In [ ]:
# Verifica que la columna 'made_sla' esté presente en las variables numéricas
if "made_sla" in numericas.columns:

    # Calcula la correlación de todas las numéricas con 'made_sla' y las ordena descendente
    corr_target = numericas.corr()["made_sla"].sort_values(ascending=False)

    # Imprime el título del análisis
    print("Correlación con made_sla:")

    # Muestra las correlaciones ordenadas
    print(corr_target)


##### Correlación de variables con `made_sla`

Este análisis identifica qué variables numéricas tienen mayor influencia lineal sobre el cumplimiento del SLA. Las variables con correlación positiva más alta son las que más contribuyen a que un ticket cumpla su SLA, mientras que las de correlación negativa más fuerte se asocian con incumplimiento. Estas variables serán candidatas prioritarias como predictores en los modelos de clasificación.


In [ ]:
# Genera histogramas para todas las variables numéricas en una cuadrícula
numericas.hist(figsize=(15, 10), bins=30)

# Ajusta el espaciado entre subgráficos para evitar superposiciones
plt.tight_layout()

# Guarda la figura generada en la ruta de gráficos
plt.savefig(f"{ruta_graficos}/Nb01_02_distribuciones_variables.png", dpi=300, bbox_inches="tight")

# Muestra los histogramas en pantalla
plt.show()


##### Análisis de distribuciones de variables numéricas

Los histogramas muestran la forma de distribución de cada variable numérica. Distribuciones asimétricas (sesgadas a izquierda o derecha) pueden requerir transformaciones (logarítmica, Box-Cox) para mejorar el rendimiento de modelos que asumen normalidad. Distribuciones multimodales (varios picos) sugieren la presencia de subgrupos en los datos. También se detectan visualmente valores atípicos extremos que podrían distorsionar los modelos.


In [ ]:
# Itera sobre una lista de columnas categóricas clave para analizar su distribución
for col in ["priority", "impact", "urgency", "incident_state", "contact_type"]:

    # Verifica que la columna exista en el DataFrame antes de procesarla
    if col in df.columns:

        # Imprime el nombre de la columna como encabezado de sección
        print(f"\n=== {col} ===")

        # Muestra el recuento de cada categoría única en la columna
        print(df[col].value_counts())


##### Análisis de variables categóricas principales

El análisis de frecuencias de las variables categóricas revela la distribución de las incidencias según prioridad, impacto, urgencia, estado y tipo de contacto. Las categorías con frecuencias muy bajas podrían agruparse para evitar sobreajuste en los modelos. La variable `incident_state` es especialmente informativa porque refleja el ciclo de vida del ticket, y valores como 'Resolved' o 'Closed' son los esperados para incidencias completadas.


In [ ]:
# Genera boxplots paralelos para todas las variables numéricas
numericas.boxplot(figsize=(15, 6), rot=90)

# Establece el título del gráfico
plt.title("Boxplots de Variables Numéricas")

# Ajusta el espaciado entre elementos del gráfico
plt.tight_layout()

# Guarda el gráfico en la ruta definida
plt.savefig(f"{ruta_graficos}/Nb01_03_boxplots.png", dpi=300, bbox_inches="tight")

# Muestra los boxplots en pantalla
plt.show()


##### Análisis de outliers mediante boxplots

Los boxplots facilitan la detección visual de valores atípicos (outliers) en las variables numéricas. Los puntos fuera de los bigotes (más allá de 1.5 veces el rango intercuartílico) son candidatos a outliers. Dependiendo del contexto del negocio, estos valores pueden ser errores de medición, eventos extremos reales o simplemente la naturaleza de la distribución. La decisión de eliminar o tratar outliers debe basarse en conocimiento del dominio y en el impacto que tengan sobre los modelos posteriores.


In [ ]:
# Imprime el encabezado del resumen de hallazgos iniciales
print("=== HALLAZGOS INICIALES ===")

# Muestra las dimensiones del dataset
print(f"1. Dataset con {df.shape[0]} registros y {df.shape[1]} columnas")

# Muestra el conteo de variables numéricas y categóricas
print(f"2. {len(numericas.columns)} variables numéricas, {len(categoricas.columns)} categóricas")

# Indica la ausencia de valores nulos significativos
print(f"3. Sin valores nulos significativos")

# Indica la ausencia de registros duplicados
print(f"4. Sin registros duplicados")

# Muestra el porcentaje de cumplimiento de SLA calculado sobre la variable objetivo
print(f"5. Target: made_sla - {round(df['made_sla'].value_counts(normalize=True)[True]*100, 2)}% cumplen SLA")


##### Resumen de hallazgos iniciales

El análisis exploratorio inicial confirma que el dataset está en condiciones adecuadas para las fases de modelado predictivo. Se destaca la ausencia de valores nulos y duplicados, lo que simplifica el preprocesamiento. La variable objetivo `made_sla` presenta un desbalanceo que deberá gestionarse durante el entrenamiento. Las correlaciones identificadas y los patrones en variables categóricas proporcionan una base sólida para la selección de características en la construcción del modelo predictivo de cumplimiento de SLA.
